# Exercise 11.3: Tag-and-probe turn-on curve

From *Programming in High Energy Particle Physics*, Chapter 11

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/isildakbora/hep-programming-guide-notebooks/blob/main/ch11/ex11_3_solution.ipynb)

Exercise 11.3 Tag-and-probe turn-on curve Perform a trigger efficiency measurement using the tag-and-probe method on \(Z \to ee\) data. Measure the efficiency as a function of electron \(p_T\) and \(\eta\), and fit the turn-on curve with an error function.

<details><summary>Hint</summary>

Start from Listing 11.3 and fit \(\varepsilon(p_T) = \tfrac{\varepsilon_{\max}}{2}\left[1 + \mathrm{erf}\!\left(\tfrac{p_T - p_T^{50}}{\sqrt{2}\,\sigma}\right)\right]\) with scipy.optimize.curve_fit , using the average of the lower and upper errors as sigma , or better, maximize a binomial likelihood of \(k\) out of \(n\) in each bin. For a 30 GeV trigger, expect \(p_T^{50}\) slightly above 30 GeV and a plateau somewhat below 100% that differs between barrel and endcap.

</details>

In [ ]:
import sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    get_ipython().run_line_magic("pip", "install -q numpy awkward vector scipy matplotlib")

**Data:** seeded Z→ee-like tag-and-probe toy events with an injected trigger response. The first electron alone explains event recording. No real CMS trigger efficiency is inferred.

### Step 1: Create tag-and-probe Z→ee pairs

In [ ]:
import numpy as np, awkward as ak, vector
from scipy.special import erf
rng=np.random.default_rng(42); vector.register_awkward(); N=30_000
probe_pt=rng.uniform(20,60,N); probe_eta=rng.uniform(-2.4,2.4,N)
tag_pt=(91.1876/2)**2/probe_pt
is_barrel=np.abs(probe_eta)<1.44
p50=np.where(is_barrel,31.5,33.0); plateau=np.where(is_barrel,.985,.925)
prob=plateau*(1+erf((probe_pt-p50)/(np.sqrt(2)*2.6)))/2
passed=rng.random(N)<prob
# Only the first electron is the triggered tag, so the probe is unbiased.
ele=ak.zip({'pt':ak.Array(np.stack([tag_pt,probe_pt],axis=1)),
            'eta':ak.Array(np.stack([np.zeros(N),probe_eta],axis=1)),
            'phi':ak.Array(np.tile([0.,np.pi],(N,1))),
            'mass':ak.Array(np.full((N,2),.000511)),
            'charge':ak.Array(np.tile([1,-1],(N,1))),
            'is_tag':ak.Array(np.tile([True,False],(N,1))),
            'trig_matched':ak.Array(np.stack([np.ones(N,dtype=bool),passed],axis=1))},with_name='Momentum4D')

### Step 2: Bin probe pass counts in pT and eta

In [ ]:
pairs=ak.argcartesian([ele,ele]); tag=ele[pairs['0']]; probe=ele[pairs['1']]
mask=(tag.is_tag)&(tag.trig_matched)&(tag.pt>35)&(tag.charge*probe.charge<0)&((tag+probe).mass>60)&((tag+probe).mass<120)
probes=probe[mask]
pt=ak.to_numpy(ak.flatten(probes.pt)); eta=ak.to_numpy(ak.flatten(probes.eta)); ok=ak.to_numpy(ak.flatten(probes.trig_matched))
assert len(pt)>1000
pt_edges=np.arange(20,62,2); eta_edges=np.array([-2.4,-1.44,0,1.44,2.4])
print('Selected probes:',len(pt))

### Step 3: Fit the turn-on by binomial likelihood

In [ ]:
from scipy.optimize import minimize
from scipy.stats import binomtest
import matplotlib.pyplot as plt
fig,ax=plt.subplots(1,2,figsize=(12,4))
for label,sel,color in [('barrel',abs(eta)<1.44,'C0'),('endcap',abs(eta)>=1.44,'C1')]:
    n,_=np.histogram(pt[sel],bins=pt_edges); k,_=np.histogram(pt[sel&ok],bins=pt_edges)
    centers=(pt_edges[1:]+pt_edges[:-1])/2; use=n>10
    def curve(x,p50,width,plateau): return plateau*(1+erf((x-p50)/(np.sqrt(2)*width)))/2
    def nll(par):
        e=np.clip(curve(centers[use],*par),1e-7,1-1e-7)
        return -np.sum(k[use]*np.log(e)+(n[use]-k[use])*np.log1p(-e))
    fit=minimize(nll,[32,3,.95],bounds=[(25,40),(.5,10),(.5,1)])
    eff=k[use]/n[use]
    ci=[binomtest(int(ki),int(ni)).proportion_ci(method='exact') for ki,ni in zip(k[use],n[use])]
    err=np.array([[e-c.low,c.high-e] for e,c in zip(eff,ci)]).T
    ax[0].errorbar(centers[use],eff,yerr=err,fmt='o',color=color,label=label)
    xx=np.linspace(20,60,200); ax[0].plot(xx,curve(xx,*fit.x),color=color)
    print(label,'pT50 [GeV]',round(fit.x[0],2),'plateau',round(fit.x[2],3))
ax[0].set(xlabel=r'Probe $p_T$ [GeV]',ylabel='Trigger efficiency',ylim=(0,1.05)); ax[0].legend()
n2,_,_=np.histogram2d(pt,eta,bins=(pt_edges,eta_edges)); k2,_,_=np.histogram2d(pt[ok],eta[ok],bins=(pt_edges,eta_edges))
e2=np.divide(k2,n2,out=np.full_like(k2,np.nan),where=n2>0)
im=ax[1].pcolormesh(pt_edges,eta_edges,e2.T,vmin=0,vmax=1); ax[1].set(xlabel=r'Probe $p_T$ [GeV]',ylabel=r'Probe $\eta$'); fig.colorbar(im,ax=ax[1],label='Efficiency'); plt.tight_layout(); plt.show()

The toy trigger has a 50% point slightly above 30 GeV and plateaus below 100%, with the endcap lower than the barrel. Real trigger efficiency requires genuine trigger-object matching and the independent tag trigger path; this toy only demonstrates the method.